## RQ3

In [ ]:
import json
import os
import numpy as np

def evaluate_prediction(pred_path, ground_truth, common_users):
    precision_list = []
    recall_list = []
    hit_ratio_list = []
    ndcg_list = []
    hit_item_list = []

    with open(pred_path, "r") as f:
        pred_dict = json.load(f)

    for user_str, pred_items in pred_dict.items():
        if user_str in common_users:
            user = int(user_str)
            if str(user) not in ground_truth:
                continue

            gt_items = set(ground_truth[str(user)])  # flat list of item_ids
            if not gt_items:
                continue

            k = len(gt_items)
            pred_topk = pred_items[:k]
            hit_items = set(pred_topk) & gt_items

            precision = len(hit_items) / k
            recall = len(hit_items) / len(gt_items)
            hit_ratio = 1.0 if len(hit_items) > 0 else 0.0

            hit_item_list.append((user, k, len(hit_items)))

            dcg = sum([1.0 / np.log2(i + 2) for i, item in enumerate(pred_topk) if item in gt_items])
            idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(gt_items), k))])
            ndcg = dcg / idcg if idcg > 0 else 0.0

            precision_list.append(precision)
            recall_list.append(recall)
            hit_ratio_list.append(hit_ratio)
            ndcg_list.append(ndcg)

    return {
        "precision": np.mean(precision_list),
        "recall": np.mean(recall_list),
        "hit_ratio": np.mean(hit_ratio_list),
        "ndcg": np.mean(ndcg_list),
        "hit_details": hit_item_list
    }

# 파일 경로
file_path = "./Augmentation/data/books"
test_path = os.path.join(file_path, "label.json")
train_path = os.path.join(file_path, "train.json")

with open(test_path, "r") as f:
    ground_truth = json.load(f)
    
with open(train_path, "r") as f:
    train_data = json.load(f)


# 공통 사용자 추출
common_users = set(train_data.keys()) & set(ground_truth.keys())

# 평가
for part in ["part5"]:
    pred_file = os.path.join(file_path, f"predict_label_part1.json")
    print(f"\n📂 Evaluating {pred_file}")
    result = evaluate_prediction(pred_file, ground_truth, common_users)

    print(f"📊 Results for {part}:")
    print(f"precision: {result['precision']:.4f}, recall: {result['recall']:.4f}, "
          f"hit_ratio: {result['hit_ratio']:.4f}, ndcg: {result['ndcg']:.4f}")


In [ ]:
# ==== Popularity vs Diversity — Predictions vs Label(GT): 진단 + 보정 + 시각화 (단일 셀) ====
import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# ---------------------------
# 0) 경로/입력 로드
# ---------------------------
file_path = "./Augmentation/data/feedback_loop"
item_attr_df = pd.read_csv(
    "./LLMRec/LLMRec_c/movielens/ml100k_llmrec_format/item_attribute.csv"
)
item_attr_df.set_index('id', inplace=True)

with open(os.path.join(file_path, "train_raw.json")) as f:
    train_data = json.load(f)

with open(os.path.join(file_path, "predict_label.json")) as f:
    pred_sample = json.load(f)

with open(os.path.join(file_path, "label.json")) as f:
    label_data = json.load(f)

# ---------------------------
# 1) item_attribute 인덱스 정규화 (0-based 강제)
# ---------------------------
idx_vals = pd.Index(item_attr_df.index).astype(int)
need_shift = (idx_vals.min() == 1) and (0 not in idx_vals)  # 1-based로 추정되면 1 빼기
if need_shift:
    item_attr_df = item_attr_df.copy()
    item_attr_df.index = (idx_vals - 1).astype(int)

# ---------------------------
# 2) 기본 통계 및 공통 사용자/인기 카운트
# ---------------------------
train_users = set(map(int, train_data.keys()))
pred_users  = set(map(int, pred_sample.keys()))
common_users = train_users & pred_users

# train 기준 아이템 등장 횟수(인기도)
item_inter_count = defaultdict(int)
for items in train_data.values():
    for it in items:
        item_inter_count[int(it)] += 1

# 전체 장르 집합/개수
all_genres = set()
for item_id in item_attr_df.index:
    g = item_attr_df.loc[item_id, 'genre']
    if isinstance(g, str):
        g = g.replace(",", "|").replace(" ", "")
        genre_list = re.split(r'[|,]', g)
        all_genres.update(gg.strip() for gg in genre_list if gg.strip())
item_genre_num = len(all_genres)

print(f"#users in train.json: {len(train_users)}")
print(f"#users in predict_label.json: {len(pred_users)}")
print(f"#common_users: {len(common_users)}")
print(f"#unique genres: {item_genre_num}")

# ---------------------------
# 3) 예측 분포 계산 (parts 리스트는 형식 보존용, 실제론 루트 predict_label.json 사용)
# ---------------------------
parts = ["part5"]  # 표기만 Ppart5로 붙여서 플롯에 색상 구분
part_results = {}

for i in parts:
    # 루트 predict_label.json 사용
    pred_data = pred_sample

    diversity_scores = {}
    popularity_scores = {}

    for user_str, rec_items in pred_data.items():
        uid = int(user_str)
        if uid not in common_users:
            continue

        genres = []
        popularity = []

        for item in rec_items:
            item_i = int(item)
            if item_i in item_attr_df.index:
                g = item_attr_df.loc[item_i, 'genre']
                if isinstance(g, str):
                    g = g.replace(",", "|").replace(" ", "")
                    genre_list = re.split(r'[|,]', g)
                    genres.extend(gg.strip() for gg in genre_list if gg.strip())
            popularity.append(item_inter_count.get(item_i, 0))

        unique_genres = set(genres)
        diversity = (len(unique_genres) / item_genre_num * 100) if item_genre_num > 0 else 0.0
        avg_popularity = (sum(popularity) / len(popularity)) if popularity else 0.0

        diversity_scores[uid] = diversity
        popularity_scores[uid] = avg_popularity

    part_df = pd.DataFrame({
        "user_id": list(diversity_scores.keys()),
        "diversity": list(diversity_scores.values()),
        "avg_popularity": list(popularity_scores.values()),
        "part": [f"P{i}"] * len(diversity_scores)
    })
    part_results[i] = part_df

# (예측) 전체 통합 DF
all_df = pd.concat(part_results.values(), ignore_index=True) if part_results else pd.DataFrame()
print(f"all_df shape (pred points): {all_df.shape}")

# ---------------------------
# 4) 실제 정답(label.json) 분포 계산 (공통 사용자 기준)
# ---------------------------
label_diversity_scores = {}
label_popularity_scores = {}

for user_str, true_items in label_data.items():
    uid = int(user_str)
    if uid not in common_users:
        continue

    genres = []
    popularity = []
    for item in true_items:
        item_i = int(item)
        if item_i in item_attr_df.index:
            g = item_attr_df.loc[item_i, 'genre']
            if isinstance(g, str):
                g = g.replace(",", "|").replace(" ", "")
                genre_list = re.split(r'[|,]', g)
                genres.extend(gg.strip() for gg in genre_list if gg.strip())
        popularity.append(item_inter_count.get(item_i, 0))

    unique_genres = set(genres)
    diversity = (len(unique_genres) / item_genre_num * 100) if item_genre_num > 0 else 0.0
    avg_popularity = (sum(popularity) / len(popularity)) if popularity else 0.0

    label_diversity_scores[uid] = diversity
    label_popularity_scores[uid] = avg_popularity

label_df = pd.DataFrame({
    "user_id": list(label_diversity_scores.keys()),
    "diversity": list(label_diversity_scores.values()),
    "avg_popularity": list(label_popularity_scores.values()),
    "part": ["Label(GT)"] * len(label_diversity_scores)
})
print(f"label_df shape: {label_df.shape}")

# ---------------------------
# 5) 병합 및 시각화
# ---------------------------
plot_df = pd.concat([all_df, label_df], ignore_index=True) if not all_df.empty else label_df.copy()
print("plot_df groups:", plot_df["part"].value_counts(dropna=False).to_dict())

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=plot_df,
    x='avg_popularity',
    y='diversity',
    hue='part',
    s=40,
    alpha=0.8
)
plt.title("Popularity vs. Diversity — Predictions vs Label(GT) (Common Users)")
plt.xlabel("Average Popularity of Items (train-based counts)")
plt.ylabel("Diversity of Items (Genre %)")
plt.grid(True)
plt.legend(title="Group")
plt.tight_layout()
plt.show()


In [ ]:
import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter

# =========================
# Paths (Augmentation)
# =========================
AUG_BASE   = "./Augmentation/data/feedback_loop"
PRED_P1    = "./Augmentation/data/feedback_loop/part1/predict_label.json"
PRED_P5    = "./Augmentation/data/feedback_loop/part5/predict_label.json"

# (장르 메타 우선순위) 1) item_attribute.csv 2) u.item
ITEM_ATTR_CSV = "./LLMRec/LLMRec_c/movielens/ml100k_llmrec_format/item_attribute.csv"
U_ITEM_PATH   = "./A-LLMRec/data/ml-100k/u.item"

# =========================
# Load genre meta (0-based id로 맞춤)
# =========================
def load_genre_meta():
    if os.path.exists(ITEM_ATTR_CSV):
        df = pd.read_csv(ITEM_ATTR_CSV)
        if "id" not in df.columns or "genre" not in df.columns:
            raise ValueError("item_attribute.csv에 'id','genre' 컬럼이 필요합니다.")
        # 인덱스가 1-based일 가능성 보정
        idx = pd.Index(df["id"]).astype(int)
        need_shift = (idx.min() == 1) and (0 not in idx)
        df = df.set_index("id")
        if need_shift:
            df = df.copy()
            df.index = (df.index.astype(int) - 1)
        # 장르를 리스트로 정규화
        gmap = {}
        for iid, g in df["genre"].items():
            if isinstance(g, str):
                g = g.replace(",", "|")
                parts = [x.strip() for x in re.split(r"[|/;,]", g) if x and str(x).strip()]
            else:
                parts = []
            gmap[int(iid)] = parts
        return gmap

    # fallback: u.item
    if not os.path.exists(U_ITEM_PATH):
        raise FileNotFoundError("장르 메타 파일을 찾을 수 없습니다.")
    genre_cols = [
        'unknown','Action','Adventure','Animation',"Children's",'Comedy','Crime',
        'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
        'Romance','Sci-Fi','Thriller','War','Western'
    ]
    col_names = ['movie_id','title','release_date','video_release_date','IMDb_URL'] + genre_cols
    meta = pd.read_csv(U_ITEM_PATH, sep='|', names=col_names, encoding='ISO-8859-1')
    # u.item은 1-based → 0-based
    meta["item_id0"] = meta["movie_id"].astype(int) - 1
    gmap = {}
    for _, r in meta.iterrows():
        genres = [g for g, f in zip(genre_cols, r[genre_cols].tolist()) if int(f) == 1]
        gmap[int(r["item_id0"])] = genres
    return gmap

GENRE_MAP = load_genre_meta()
ALL_GENRES = sorted({g for lst in GENRE_MAP.values() for g in lst})
MAX_GENRES = max(1, len(ALL_GENRES))  # 0-division 방지

# =========================
# Popularity (train.json에서 아이템 등장 횟수)
# =========================
with open(os.path.join(AUG_BASE, "train.json")) as f:
    train_data = json.load(f)
train_users = set(map(int, train_data.keys()))

item_pop = Counter()
for items in train_data.values():
    for it in items:
        item_pop[int(it)] += 1

# =========================
# Helpers
# =========================
def load_predictions_json(path):
    """predict_label.json -> {uid(int): [item_id(int), ...]}"""
    with open(path) as f:
        raw = json.load(f)
    out = {}
    for u_str, items in raw.items():
        try:
            uid = int(u_str)
        except:
            continue
        # 값이 리스트가 아닐 수도 있으니 방어
        if isinstance(items, list):
            out[uid] = [int(x) for x in items]
        else:
            try:
                out[uid] = [int(items)]
            except:
                out[uid] = []
    return out

def metrics_from_item_ids(user_items: dict[int, list[int]]):
    """
    Return: dict[uid] -> (avg_popularity, diversity_pct)
    - popularity: mean(item_pop[item_id])
    - diversity: unique genre count / MAX_GENRES * 100
    """
    out = {}
    for uid, items in user_items.items():
        ids = [int(x) for x in items]
        pops = [item_pop.get(it, 0) for it in ids]
        genres = []
        for it in ids:
            genres.extend(GENRE_MAP.get(int(it), []))
        uniq = len(set(genres))
        avg_pop = float(np.mean(pops)) if pops else 0.0
        div_pct = (uniq / MAX_GENRES) * 100.0
        out[uid] = (avg_pop, div_pct)
    return out

def common_users_among(*user_sets):
    return set.intersection(*user_sets) if user_sets else set()

# =========================
# Load predictions for part1 & part5
# =========================
pred_p1 = load_predictions_json(PRED_P1)
pred_p5 = load_predictions_json(PRED_P5)

users_p1 = set(pred_p1.keys())
users_p5 = set(pred_p5.keys())
common_users = common_users_among(train_users, users_p1, users_p5)

if not common_users:
    raise RuntimeError("train ∩ part1 ∩ part5 공통 사용자가 없습니다.")

# 공통 사용자만 필터
pred_p1_common = {u: pred_p1[u] for u in common_users}
pred_p5_common = {u: pred_p5[u] for u in common_users}

# =========================
# Compute metrics per part
# =========================
m_p1 = metrics_from_item_ids(pred_p1_common)
m_p5 = metrics_from_item_ids(pred_p5_common)

# 유저별 궤적 dict
traj = {u: {"pop": [], "div": []} for u in common_users}
for u in common_users:
    p1_pop, p1_div = m_p1[u]
    p5_pop, p5_div = m_p5[u]
    traj[u]["pop"] = [p1_pop, p5_pop]
    traj[u]["div"] = [p1_div, p5_div]

# =========================
# Δ(Part1→Part5) DataFrame
# =========================
dx = [traj[u]["pop"][1] - traj[u]["pop"][0] for u in common_users]
dy = [traj[u]["div"][1] - traj[u]["div"][0] for u in common_users]

df_delta = pd.DataFrame({
    "delta": dx + dy,
    "metric": (["Popularity"] * len(dx)) + (["Diversity"] * len(dy)),
    "transition": ["part1 - part5"] * (len(dx) + len(dy)),
    "case": ["Augment"] * (len(dx) + len(dy)),
})

# =========================
# 시각화
# =========================
sns.set_style("whitegrid")
plt.figure(figsize=(8,6))
ax = sns.boxplot(data=df_delta, x="metric", y="delta", palette="Set2")
ax.set_title("Popularity & Diversity Change — part1 - part5 (Augmentation predictions)")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
ax.axhline(0, linestyle="--", linewidth=1, alpha=0.6)
plt.tight_layout()
plt.show()

# (선택) 요약 통계 출력
print("Summary (part1→part5)")
print(df_delta.groupby("metric")["delta"].describe().round(3))


## RQ1, 2

In [ ]:
import pickle
import os

# 파일 경로 설정 (이전 코드의 저장 경로 기준)
file_path = "./Augmentation/data/books/aug_triplets_part1_step0.pkl"

if os.path.exists(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)

    print(f"📂 파일 로드 완료: {file_path}")
    print(f"📊 데이터 타입: {type(data)}")
    print(f"🔢 총 Triplets 개수: {len(data)}")
    print("=" * 40)
    print("👀 상위 20개 샘플 (User ID, Pos Item, Neg Item):")
    
    # 데이터가 리스트인지 확인 후 출력
    if isinstance(data, list):
        for idx, item in enumerate(data[:20]):
            print(f"{idx+1}: {item}")
    else:
        print(data)
else:
    print(f"❌ 파일을 찾을 수 없습니다: {file_path}")

In [ ]:
# === 기본 설정 ===
import os, re, json, gzip, pickle, glob, math
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "./Augmentation"                    # 필요시 수정
SAVE_DIR = f"{BASE_DIR}/data/feedback_loop"                        # train.json, label.json, pkl 저장 위치
ML100K_DIR = f"{BASE_DIR}/data/ml-100k"                            # MovieLens-100k 원본(u.item 등) 위치

# 노트북 출력 옵션
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def try_read_json(path: str, default=None):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return default


# aug_triplets 파일들 수집
pkl_paths = sorted(glob.glob(f"{SAVE_DIR}/part1/aug_triplets_part*_step*.pkl"))
print(f"Found {len(pkl_paths)} files")

# 모든 step의 triplets 병합 → DataFrame(u, pos, neg, part, step)
rows = []
for p in pkl_paths:
    triplets = load_pickle(p)
    # 파일명에서 part, step 추출
    m = re.search(r"part(\d+)_step(\d+)", os.path.basename(p))
    part = int(m.group(1)) if m else None
    step = int(m.group(2)) if m else None
    
    for (u, pos, neg) in triplets:
        rows.append({"u": int(u), "pos": int(pos), "neg": int(neg), "part": part, "step": step})

aug_df = pd.DataFrame(rows)
print(aug_df.shape)
print(aug_df.head())


# MovieLens-100k u.item: pipe('|') 구분.
# 컬럼: movie id | movie title | release date | video release | IMDb URL | [19 genres flags]
# id는 1-based. 너의 파이프라인은 0-based를 쓰므로 -1 처리.
def load_ml100k_genres(u_item_path: str) -> pd.DataFrame:
    # 장르 이름들 (ML-100k 고정)
    genre_cols = [
        "unknown","Action","Adventure","Animation","Children's","Comedy","Crime",
        "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
        "Romance","Sci-Fi","Thriller","War","Western"
    ]
    names = ["movie_id","title","release_date","video_release_date","imdb_url"] + genre_cols
    df = pd.read_csv(u_item_path, sep="|", header=None, names=names, encoding="latin-1")
    df["item_id"] = df["movie_id"].astype(int) - 1  # 0-based로 변환
    print(df.head(10))
    # long form로 펼치기: (item_id, genre) where flag==1
    g_long = []
    for _, r in df.iterrows():
        for g in genre_cols:
            if int(r[g]) == 1:
                g_long.append((r["item_id"], g))
    gdf = pd.DataFrame(g_long, columns=["item_id","genre"])
    return gdf

# 보조: json.gz 메타에서 genre 키를 탐색
def load_aux_genres_from_meta(meta_gz_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(meta_gz_path):
        return None
    with gzip.open(meta_gz_path, "rb") as f:
        raw = f.read()
    try:
        meta = json.loads(raw.decode("utf-8"))
    except Exception:
        meta = pickle.loads(raw)
    # 가능한 키 후보들
    cand_keys = ["genre","genres","category","categories"]
    found = None
    for k in cand_keys:
        v = meta.get(k)
        if isinstance(v, dict) and len(v)>0:
            found = v
            break
    if found is None:
        return None
    # 값이 list 또는 str라고 가정
    rows = []
    for sid, val in found.items():
        try:
            iid = int(sid)
        except:
            continue
        if isinstance(val, list):
            gs = [str(x) for x in val]
        else:
            gs = re.split(r"[;,/|]", str(val))  # 단순 분할
        for g in [x.strip() for x in gs if x and str(x).strip()]:
            rows.append((iid, g))
    return pd.DataFrame(rows, columns=["item_id","genre"])

# 통합 로딩
u_item_path = f"{ML100K_DIR}/u.item"
meta_gz_path = f"{ML100K_DIR}/ml-100k_text_name_dict.json.gz"

genre_df = None
if os.path.exists(u_item_path):
    genre_df = load_ml100k_genres(u_item_path)
else:
    genre_df = load_aux_genres_from_meta(meta_gz_path)

if genre_df is None or genre_df.empty:
    print("장르 정보를 찾지 못했습니다. 이후 장르 기반 분석은 건너뜁니다.")
else:
    print(genre_df.shape, genre_df.head())
    
# === predict_label.json 장르 분포(요약) 추가 ===
PRED_PATH = os.path.join(SAVE_DIR, "part1/predict_label.json")

def _genre_counts_quick(item_series: pd.Series, genre_df: pd.DataFrame) -> pd.Series:
    if item_series is None or item_series.empty:
        return pd.Series(dtype=int)
    return (item_series.to_frame("item_id")
            .merge(genre_df, on="item_id", how="left")
            .dropna(subset=["genre"])
            .groupby("genre").size().sort_values(ascending=False))

pred_p = None  # 두 번째 블록에서 재사용 가능하게 외부 변수에 둠
if os.path.exists(PRED_PATH) and genre_df is not None and not genre_df.empty:
    with open(PRED_PATH, "r") as f:
        pred_dict = json.load(f)

    # 추천 아이템 id 모두 펼치기
    pred_items = []
    for _, items in pred_dict.items():
        if isinstance(items, list):
            pred_items.extend(int(x) for x in items)
        else:
            try:
                pred_items.append(int(items))
            except Exception:
                pass

    pred_series = pd.Series(pred_items, dtype="Int64").dropna().astype(int)
    pred_counts = _genre_counts_quick(pred_series, genre_df)

    # 확률로 정규화
    pred_p = pred_counts / max(1, pred_counts.sum())

    # 요약 출력
    print(f"[predict_label] total items counted: {int(pred_counts.sum())}")
    print(pred_counts.head(10))
else:
    print("[predict_label] 파일이 없거나 genre_df가 비어있어 예측 분포 계산을 건너뜁니다.")


### 장르분포

In [ ]:
def genre_counts(df_items: pd.Series, genre_df: pd.DataFrame) -> pd.Series:
    # df_items: 아이템 id 시리즈
    # genre_df: (item_id, genre)
    return (df_items.to_frame("item_id")
            .merge(genre_df, on="item_id", how="left")
            .dropna(subset=["genre"])
            .groupby("genre").size().sort_values(ascending=False))

if genre_df is not None and not genre_df.empty:
    pos_counts = genre_counts(aug_df["pos"], genre_df)
    neg_counts = genre_counts(aug_df["neg"], genre_df)

    # 전체 카탈로그 분포(비교 기준)
    catalog_counts = genre_df.groupby("genre").size()

    def to_prob(s: pd.Series) -> pd.Series:
        s = s[s.index.notnull()]
        return (s / s.sum()).reindex(sorted(set(s.index) | set(catalog_counts.index))).fillna(0)

    pos_p = to_prob(pos_counts)
    neg_p = to_prob(neg_counts)
    cat_p = to_prob(catalog_counts)

    # === predict_label.json 분포 추가 ===
    PRED_PATH = os.path.join(SAVE_DIR, "predict_label.json")
    # 1) 블록1에서 pred_p를 이미 만들었다면 재사용
    if 'pred_p' in globals() and isinstance(pred_p, pd.Series) and not pred_p.empty:
        pred_p2 = (pred_p / max(1, pred_p.sum())).reindex(sorted(set(cat_p.index) | set(pred_p.index))).fillna(0)
    else:
        # 2) 블록1을 안 돌렸다면 여기서 다시 계산
        pred_p2 = None
        if os.path.exists(PRED_PATH):
            with open(PRED_PATH, "r") as f:
                pred_dict = json.load(f)
            pred_items = []
            for _, items in pred_dict.items():
                if isinstance(items, list):
                    pred_items.extend(int(x) for x in items)
                else:
                    try:
                        pred_items.append(int(items))
                    except Exception:
                        pass
            if len(pred_items) > 0:
                pred_series = pd.Series(pred_items, dtype="Int64").dropna().astype(int)
                pred_counts = genre_counts(pred_series, genre_df)
                pred_p2 = (pred_counts / max(1, pred_counts.sum())).reindex(
                    sorted(set(cat_p.index) | set(pred_counts.index))
                ).fillna(0)

    # 엔트로피/ KL
    def entropy(p):
        p = p[p>0].values
        return float(-(p * np.log(p)).sum())
    def kl(p, q):
        eps = 1e-12
        p = p.values + eps
        q = q.values + eps
        return float((p * np.log(p/q)).sum())

    metrics = {
        "H(pos)": entropy(pos_p),
        "H(neg)": entropy(neg_p),
        "KL(pos || real)": kl(pos_p, cat_p),
        "KL(neg || real)": kl(neg_p, cat_p),
    }
    # pred 지표도 가능하면 추가
    if pred_p2 is not None:
        # cat와 인덱스 정렬 일치
        cat_aligned = cat_p.reindex(sorted(set(cat_p.index) | set(pred_p2.index))).fillna(0)
        metrics["H(pred)"] = entropy(pred_p2)
        metrics["KL(pred || real)"] = kl(pred_p2, cat_aligned)

    print(metrics)

    # 막대 그래프 (pred 포함)
    df_plot = pd.DataFrame({
        "pos": pos_p,
        "neg": neg_p,
        "real": cat_p
    })
    if pred_p2 is not None:
        # 인덱스 합집합으로 맞춰 병합
        all_idx = sorted(set(df_plot.index) | set(pred_p2.index))
        df_plot = df_plot.reindex(all_idx).fillna(0)
        df_plot["pred"] = pred_p2.reindex(all_idx).fillna(0)

    ax = df_plot.sort_values("real", ascending=False).plot(kind="bar", figsize=(12,5))
    ax.set_title("Genre distribution (probabilities) — pos / neg / real" + (" / pred" if pred_p2 is not None else ""))
    ax.set_ylabel("Probability")
    plt.tight_layout()
    plt.show()


In [ ]:
# === 스텝별 pos/neg + 정답(라벨, 사용자기준) + 기준(카탈로그/실제) 분포를 한 번에 시각화 ===
import json, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import ceil

# 필수: aug_df( u,pos,neg,step ), genre_df( item_id,genre ), SAVE_DIR, ML100K_DIR 가 준비되어 있어야 함.

# 1) 보조 유틸
def to_prob(s: pd.Series) -> pd.Series:
    s = s[s.index.notnull()]
    tot = s.sum()
    if tot == 0:
        return s.astype(float)
    return s / tot

def genre_counts_from_items(item_series: pd.Series, genre_df: pd.DataFrame) -> pd.Series:
    if item_series is None or len(item_series) == 0:
        return pd.Series(dtype=int)
    return (
        item_series.to_frame("item_id")
        .merge(genre_df, on="item_id", how="left")
        .dropna(subset=["genre"])
        .groupby("genre").size().sort_values(ascending=False)
    )

# 2) 기준 분포 준비
# 2-1) 카탈로그: 아이템 균등(아이템 개수 기준)
catalog_counts = genre_df.groupby("genre")["item_id"].nunique().sort_values(ascending=False)
catalog_p = to_prob(catalog_counts)

# 2-2) 실제(상호작용): label.json

with open(f"{SAVE_DIR}/label.json","r") as f:
    label_dict = json.load(f)
label_dict = {int(k): [int(x) for x in v] for k,v in (label_dict or {}).items()}


# 3) (선택) ground_truth(date_idx) 기반 per-step 정답 분포를 쓰고 싶다면, 아래 파일을 직접 저장해 두고 사용
# try:
#     import pickle, gzip
#     with gzip.open(f"{SAVE_DIR}/ground_truth.pkl.gz","rb") as f:
#         gtz = pickle.load(f)  # {"ground_truth": {u: [(item_id,date_idx),...]}, "step_time_ranges": {step: set_of_date_idx}}
#     ground_truth = gtz["ground_truth"]
#     step_time_ranges = gtz["step_time_ranges"]
#     use_true_step_labels = True
# except Exception:
#     use_true_step_labels = False

# 4) 스텝별 분포 계산
steps = sorted(aug_df["step"].dropna().unique().tolist())
if len(steps) == 0:
    raise RuntimeError("aug_df에 step 정보가 없습니다.")

per_step_records = []
for s in steps:
    sdf = aug_df[aug_df["step"] == s]
    # pos/neg 분포
    pos_p = to_prob(genre_counts_from_items(sdf["pos"], genre_df))
    neg_p = to_prob(genre_counts_from_items(sdf["neg"], genre_df))

    # 정답(라벨) 분포 — 사용자 기준 근사:
    # 이 step에서 증강이 일어난 사용자들의 label 아이템을 모두 모아 분포 계산
    users_in_step = sdf["u"].unique().tolist()
    label_items_step = []
    for u in users_in_step:
        label_items_step.extend(label_dict.get(int(u), []))
    true_p = to_prob(genre_counts_from_items(pd.Series(label_items_step, dtype=int), genre_df))

    # (선택) ground_truth로 정확한 step 타임슬라이스를 쓰는 경우:
    # if use_true_step_labels:
    #     dset = set()
    #     for u, pairs in ground_truth.items():
    #         for (iid, t) in pairs:
    #             if t in step_time_ranges.get(int(s), set()):
    #                 dset.add(iid)
    #     true_p = to_prob(genre_counts_from_items(pd.Series(list(dset), dtype=int), genre_df))

    # 공통 장르 인덱스 맞춤
    all_idx = sorted(set(pos_p.index) | set(neg_p.index) | set(true_p.index) | set(catalog_p.index))
    per_step_records.append({
        "step": int(s),
        "pos_p": pos_p.reindex(all_idx).fillna(0.0),
        "neg_p": neg_p.reindex(all_idx).fillna(0.0),
        "label_p": true_p.reindex(all_idx).fillna(0.0),
        "real_item_distribution": catalog_p.reindex(all_idx).fillna(0.0),
        "genres": all_idx,
    })

# 5) 시각화 — 하나의 그림에 스텝별 서브플롯(행렬)로 배치
n = len(per_step_records)
cols = 2
rows = ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(14, 4*rows), sharey=True)
if rows == 1 and cols == 1:
    axes = np.array([[axes]])
elif rows == 1:
    axes = np.array([axes])
axes = axes.reshape(rows, cols)

for i, rec in enumerate(per_step_records):
    r, c = divmod(i, cols)
    ax = axes[r, c]

    df_plot = pd.DataFrame({
        "pos": rec["pos_p"],
        "neg": rec["neg_p"],
        "gt(labels)": rec["label_p"],
        "real_item_distribution": rec["real_item_distribution"]
    }, index=rec["genres"])

    # 장르가 너무 많으면 상위 K만 보기 (옵션)
    # K = 15
    # keep = df_plot["interaction"].sort_values(ascending=False).head(K).index
    # df_plot = df_plot.loc[keep]

    df_plot.plot(kind="bar", ax=ax, width=0.85)
    ax.set_title(f"Step {rec['step']}")
    ax.set_ylabel("Probability")
    ax.legend(loc="upper right", fontsize=9)
    ax.tick_params(axis='x', labelrotation=75)

# 빈 축 숨기기
for j in range(n, rows*cols):
    r, c = divmod(j, cols)
    axes[r, c].axis("off")

fig.suptitle("Genre distributions per step — pos / neg / gt(labels) / catalog / interaction", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


## RQ4

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# =========================
# 경로/설정
# =========================
AUG_BASE = "./Augmentation/data/feedback_loop"
PARTS    = 1                          # 피드백 루프 step 수 (0..PARTS-1)
STEPS    = list(range(PARTS))
CLUSTER_STEP = STEPS[-1]              # 최종 step에서 KMeans 수행
OUT_DIR  = os.path.join(AUG_BASE, "figs_tsne_cluster_progress")
os.makedirs(OUT_DIR, exist_ok=True)

TSNE_PERPLEXITY      = 30
TSNE_RANDOM_STATE    = 42
KMEANS_RANDOM_STATE  = 42
K_USERS = 2
K_ITEMS = 2

# (선택) t-SNE 가속 샘플링
SUBSAMPLE_USERS_FOR_TSNE = None  # 예: 5000
SUBSAMPLE_ITEMS_FOR_TSNE = None  # 예: 5000

# =========================
# 유틸
# =========================
def load_users_items_for_step(base_dir: str, parts: int, step: int):
    upath = os.path.join(base_dir, f"embeddings_users_part{parts}_step{step}.npy")
    ipath = os.path.join(base_dir, f"embeddings_items_part{parts}_step{step}.npy")
    if not os.path.exists(upath):
        raise FileNotFoundError(upath)
    U = np.load(upath)  # shape: (num_users_step, d)
    I = np.load(ipath) if os.path.exists(ipath) else None  # shape: (num_items_step, d) or None
    if U.ndim != 2:
        raise ValueError(f"user_emb shape invalid at step{step}: {U.shape}")
    if I is not None and I.ndim != 2:
        raise ValueError(f"item_emb shape invalid at step{step}: {I.shape}")
    return U, I

def load_common_users_from_json(base_dir: str):
    train_path = os.path.join(base_dir, "train_raw.json")
    label_path = os.path.join(base_dir, "label.json")
    with open(train_path, "r") as f:
        train = json.load(f)
    with open(label_path, "r") as f:
        label = json.load(f)
    train_users = set(map(int, train.keys()))
    label_users = set(map(int, label.keys()))
    return np.array(sorted(train_users & label_users), dtype=np.int64)

def align_users_for_step(common_users: np.ndarray, num_users_step: int):
    """해당 step의 사용자 수 범위(0..num_users_step-1) 안에 있는 공통 유저만 유지"""
    if len(common_users) == 0:
        return np.array([], dtype=np.int64)
    mask = (common_users >= 0) & (common_users < num_users_step)
    return common_users[mask]

# =========================
# 1) 각 step 임베딩 로드
# =========================
print("Loading augmentation embeddings per step...")
user_by_step = {}
item_by_step = {}
dims = set()

for s in STEPS:
    U, I = load_users_items_for_step(AUG_BASE, PARTS, s)
    user_by_step[s] = U
    item_by_step[s] = I
    dims.add(U.shape[1])
    if I is not None:
        dims.add(I.shape[1])
    print(f"  - step{s}: users={U.shape}, items={None if I is None else I.shape}")

if len(dims) != 1:
    raise RuntimeError(f"임베딩 차원이 step별로 다릅니다: {dims}")
D = dims.pop()

# =========================
# 2) train∩label 공통 유저 산출 + step별 정렬
# =========================
print("Compute base common users from train ∩ label")
common_users_base = load_common_users_from_json(AUG_BASE)
if len(common_users_base) == 0:
    raise RuntimeError("train ∩ label 공통 유저가 없습니다.")
print(f"  - base common users: {len(common_users_base)}")

aligned_users_by_step = {s: align_users_for_step(common_users_base, user_by_step[s].shape[0])
                         for s in STEPS}
print({s: len(aligned_users_by_step[s]) for s in STEPS})

# =========================
# 3) 아이템 공통 부분 정렬(최소 길이)
# =========================
item_lengths = [item_by_step[s].shape[0] for s in STEPS if item_by_step[s] is not None]
if len(item_lengths) == 0:
    I_base = 0
else:
    I_base = min(item_lengths)
    uniq = set(item_lengths)
    if len(uniq) != 1:
        print(f"[warn] step별 아이템 수가 다릅니다: {uniq} → 앞에서 {I_base}개만 공통 사용")
    for s in STEPS:
        I = item_by_step[s]
        item_by_step[s] = None if I is None or I.shape[0] < I_base else I[:I_base]

rng = np.random.RandomState(0)
if I_base > 0:
    if SUBSAMPLE_ITEMS_FOR_TSNE is not None and SUBSAMPLE_ITEMS_FOR_TSNE < I_base:
        keep_items = np.sort(rng.choice(I_base, size=SUBSAMPLE_ITEMS_FOR_TSNE, replace=False))
    else:
        keep_items = np.arange(I_base)
else:
    keep_items = np.array([], dtype=int)

# =========================
# 4) 최종 step에서 KMeans → 레이블 생성
# =========================
print(f"KMeans on final step: step{CLUSTER_STEP}")
U_final = user_by_step[CLUSTER_STEP]
ids_final = aligned_users_by_step[CLUSTER_STEP]
if len(ids_final) < K_USERS:
    raise RuntimeError(f"최종 step에서 클러스터링 가능한 공통 유저가 부족합니다: {len(ids_final)}")

emb_final_users = U_final[ids_final, :]  # (|ids_final|, D)
user_kmeans = KMeans(n_clusters=K_USERS, random_state=KMEANS_RANDOM_STATE, n_init=10)
user_labels_final = user_kmeans.fit_predict(emb_final_users)
user_center_idx = {k: user_labels_final == k for k in range(K_USERS)}

# 아이템 KMeans (최종 step 공통 아이템)
if item_by_step[CLUSTER_STEP] is not None and len(keep_items) > 0:
    I_final = item_by_step[CLUSTER_STEP][keep_items]  # (|keep_items|, D)
    item_kmeans = KMeans(n_clusters=K_ITEMS, random_state=KMEANS_RANDOM_STATE, n_init=10)
    item_labels_final = item_kmeans.fit_predict(I_final)
else:
    item_kmeans = None
    item_labels_final = None
    print("[warn] 최종 step 아이템이 없어 item clustering 생략")

# =========================
# 5) t-SNE에 넣을 데이터 블록 구성 (users + items across steps)
# =========================
print("Preparing t-SNE blocks...")
ids_for_tsne = {}      # step -> user_id 배열(샘플링 반영)
users_for_tsne = {}    # step -> user_emb 배열
item_for_tsne  = {}    # step -> item_emb 배열 (공통 keep_items에 해당)

# 유저 샘플링 기준: 최종 step의 ids_final 집합
if SUBSAMPLE_USERS_FOR_TSNE is not None and SUBSAMPLE_USERS_FOR_TSNE < len(ids_final):
    sample_users = np.sort(rng.choice(ids_final, size=SUBSAMPLE_USERS_FOR_TSNE, replace=False))
else:
    sample_users = ids_final

for s in STEPS:
    ids_s = aligned_users_by_step[s]
    if len(ids_s) == 0:
        ids_for_tsne[s] = np.array([], dtype=np.int64)
        users_for_tsne[s] = np.empty((0, D), dtype=np.float32)
    else:
        mask = np.isin(ids_s, sample_users)
        ids_for_tsne[s] = ids_s[mask]
        users_for_tsne[s] = user_by_step[s][ids_for_tsne[s], :]

    I = item_by_step[s]
    item_for_tsne[s] = None if I is None or len(keep_items) == 0 else I[keep_items]

# =========================
# 6) t-SNE 수행 (모든 step의 유저/아이템 합쳐서)
# =========================
print("Fitting joint t-SNE on users + items across steps")
blocks = []
slices = {}  # step -> {"user": (st,en), "item": (st,en) or None}
cursor = 0

for s in STEPS:
    Ublk = users_for_tsne[s]
    blocks.append(Ublk)
    u_st, u_en = cursor, cursor + len(Ublk)
    cursor = u_en

    Iblk = item_for_tsne[s]
    if Iblk is not None and len(Iblk) > 0:
        blocks.append(Iblk)
        i_st, i_en = cursor, cursor + len(Iblk)
        cursor = i_en
        slices[s] = {"user": (u_st, u_en), "item": (i_st, i_en)}
    else:
        slices[s] = {"user": (u_st, u_en), "item": None}

if len(blocks) == 0 or sum(len(b) for b in blocks) < 3:
    raise RuntimeError("t-SNE에 사용할 표본이 너무 적습니다.")

X_all = np.vstack(blocks)
perp = min(TSNE_PERPLEXITY, max(5, (len(X_all) - 1)//3))
tsne = TSNE(
    n_components=2,
    perplexity=perp,
    init="pca",
    learning_rate="auto",
    random_state=TSNE_RANDOM_STATE,
    max_iter=1000,
    verbose=1
)
X_tsne = tsne.fit_transform(X_all)

# 좌표 범위(공통 축)
xmin, ymin = X_tsne.min(axis=0)
xmax, ymax = X_tsne.max(axis=0)
xpad = 0.05 * (xmax - xmin) if xmax > xmin else 0.5
ypad = 0.05 * (ymax - ymin) if ymax > ymin else 0.5

# 최종 step 중심 좌표(사용자/아이템)
palette = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple",
           "tab:brown","tab:pink","tab:gray","tab:olive","tab:cyan"]

def compute_centers(coords: np.ndarray, labels: np.ndarray, K: int):
    centers = {}
    for k in range(K):
        mask = (labels == k)
        if np.any(mask):
            centers[k] = coords[mask].mean(axis=0)
    return centers

# 최종 step 사용자/아이템 좌표 슬라이스
u_st_f, u_en_f = slices[CLUSTER_STEP]["user"]
coords_u_final = X_tsne[u_st_f:u_en_f]
# user_labels_final는 ids_final(=샘플링 전 최종 step 공통유저)에 대한 레이블
# t-SNE에는 sample_users만 포함됐을 수 있으므로, ids_for_tsne[CLUSTER_STEP] 순서에 맞춰 레이블 재정렬
ids_tsne_final = ids_for_tsne[CLUSTER_STEP]
# ids_tsne_final의 각 user_id가 ids_final 내에서의 index를 찾아 매핑
idx_map_final = {int(u): i for i, u in enumerate(ids_final.tolist())}
labels_u_for_plot = np.array([user_labels_final[idx_map_final[int(u)]] for u in ids_tsne_final], dtype=int)
user_centers = compute_centers(coords_u_final, labels_u_for_plot, K_USERS)

if slices[CLUSTER_STEP]["item"] is not None and item_labels_final is not None:
    i_st_f, i_en_f = slices[CLUSTER_STEP]["item"]
    coords_i_final = X_tsne[i_st_f:i_en_f]
    item_centers = compute_centers(coords_i_final, item_labels_final, K_ITEMS)
else:
    item_centers = {}

# =========================
# 7) 그림 저장 — Users only / Items only / Users+Items
# =========================
def plot_users_only():
    S = len(STEPS)
    cols = min(3, S)
    rows = math.ceil(S / cols)
    fig = plt.figure(figsize=(5*cols, 4*rows))
    for idx, s in enumerate(STEPS, start=1):
        ax = fig.add_subplot(rows, cols, idx)
        u_st, u_en = slices[s]["user"]
        coords_u = X_tsne[u_st:u_en]
        # 최종 step 기준 라벨로 색칠
        if s == CLUSTER_STEP:
            labels_plot = labels_u_for_plot
        else:
            # 다른 step은 같은 사용자(샘플링된 ids)에 대해 동일한 라벨을 사용
            # ids_for_tsne[s] -> ids_tsne_final로 매핑
            ids_tsne_s = ids_for_tsne[s]
            labels_plot = np.empty((len(ids_tsne_s),), dtype=int)
            for i, uid in enumerate(ids_tsne_s):
                labels_plot[i] = labels_u_for_plot[np.where(ids_tsne_final == uid)[0][0]]
        for k in range(K_USERS):
            mk = (labels_plot == k)
            if np.any(mk):
                ax.scatter(coords_u[mk,0], coords_u[mk,1], s=8, alpha=0.7,
                           c=palette[k % len(palette)], label=f"Users: C{k}")
        if s == CLUSTER_STEP:
            for k, ctr in user_centers.items():
                ax.scatter(ctr[0], ctr[1], s=120, marker="X",
                           c=palette[k % len(palette)], edgecolor="k")
        ax.set_title(f"step{s} (Users only, colored by step{CLUSTER_STEP})")
        ax.set_xticks([]); ax.set_yticks([])
        # 고정 축을 쓰고 싶으면 주석 해제
        # ax.set_xlim(xmin - xpad, xmax + xpad)
        # ax.set_ylim(ymin - ypad, ymax + ypad)
        if idx == 1:
            ax.legend(loc="best", frameon=True)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_tsne2d_USERS_ONLY.png")
    plt.savefig(out, dpi=220); plt.close()
    print("  - saved:", out)

def plot_items_only():
    has_any = any(item_for_tsne[s] is not None for s in STEPS) and (item_labels_final is not None)
    if not has_any:
        print("  - Item only plot 생략(아이템 임베딩/라벨 없음)")
        return
    S = len(STEPS)
    cols = min(3, S)
    rows = math.ceil(S / cols)
    fig = plt.figure(figsize=(5*cols, 4*rows))
    for idx, s in enumerate(STEPS, start=1):
        ax = fig.add_subplot(rows, cols, idx)
        if slices[s]["item"] is not None:
            i_st, i_en = slices[s]["item"]
            coords_i = X_tsne[i_st:i_en]
            # 최종 step의 item_labels_final을 그대로 사용 (공통 keep_items에 대응)
            for k in range(K_ITEMS):
                mk = (item_labels_final == k)
                if np.any(mk):
                    ax.scatter(coords_i[mk,0], coords_i[mk,1], s=5, alpha=0.35,
                               c=palette[(k+2) % len(palette)], label=f"Items: C{k}")
            if s == CLUSTER_STEP and len(item_centers) > 0:
                for k, ctr in item_centers.items():
                    ax.scatter(ctr[0], ctr[1], s=110, marker="D",
                               c=palette[(k+2) % len(palette)], edgecolor="k")
        ax.set_title(f"step{s} (Items only, colored by step{CLUSTER_STEP})")
        ax.set_xticks([]); ax.set_yticks([])
        if idx == 1:
            ax.legend(loc="best", frameon=True)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_tsne2d_ITEMS_ONLY.png")
    plt.savefig(out, dpi=220); plt.close()
    print("  - saved:", out)

def plot_users_items():
    S = len(STEPS)
    cols = min(3, S)
    rows = math.ceil(S / cols)
    fig = plt.figure(figsize=(5*cols, 4*rows))
    for idx, s in enumerate(STEPS, start=1):
        ax = fig.add_subplot(rows, cols, idx)
        # 아이템
        if slices[s]["item"] is not None and item_labels_final is not None:
            i_st, i_en = slices[s]["item"]
            coords_i = X_tsne[i_st:i_en]
            for k in range(K_ITEMS):
                mk = (item_labels_final == k)
                if np.any(mk):
                    ax.scatter(coords_i[mk,0], coords_i[mk,1], s=5, alpha=0.35,
                               c=palette[(k+2) % len(palette)], label=f"Items: C{k}")
            if s == CLUSTER_STEP and len(item_centers) > 0:
                for k, ctr in item_centers.items():
                    ax.scatter(ctr[0], ctr[1], s=110, marker="D",
                               c=palette[(k+2) % len(palette)], edgecolor="k")
        # 유저
        u_st, u_en = slices[s]["user"]
        coords_u = X_tsne[u_st:u_en]
        # step별 유저 라벨 구성
        if s == CLUSTER_STEP:
            labels_plot = labels_u_for_plot
        else:
            ids_tsne_s = ids_for_tsne[s]
            labels_plot = np.empty((len(ids_tsne_s),), dtype=int)
            for i, uid in enumerate(ids_tsne_s):
                labels_plot[i] = labels_u_for_plot[np.where(ids_tsne_final == uid)[0][0]]
        for k in range(K_USERS):
            mk = (labels_plot == k)
            if np.any(mk):
                ax.scatter(coords_u[mk,0], coords_u[mk,1], s=8, alpha=0.7,
                           c=palette[k % len(palette)], label=f"Users: C{k}")
        if s == CLUSTER_STEP:
            for k, ctr in user_centers.items():
                ax.scatter(ctr[0], ctr[1], s=120, marker="X",
                           c=palette[k % len(palette)], edgecolor="k")

        ax.set_title(f"step{s} (Users & Items, colored by step{CLUSTER_STEP})")
        ax.set_xlim(xmin - xpad, xmax + xpad)
        ax.set_ylim(ymin - ypad, ymax + ypad)
        ax.set_xticks([]); ax.set_yticks([])
        if idx == 1:
            ax.legend(loc="best", frameon=True)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_tsne2d_USERS_ITEMS.png")
    plt.savefig(out, dpi=220); plt.close()
    print("  - saved:", out)

print("Plotting...")
plot_users_only()
plot_items_only()
plot_users_items()
print("Done.")


In [ ]:
# ===== 9) 클러스터 중심 간 거리(step별) 계산 & 시각화 =====
import csv
import numpy as np
import matplotlib.pyplot as plt
import os
from itertools import combinations

# 전제: 위에서 다음 변수들이 이미 정의되어 있음
# - STEPS, CLUSTER_STEP, OUT_DIR
# - X_tsne, slices, ids_for_tsne, item_for_tsne
# - user_by_step, item_by_step, aligned_users_by_step
# - ids_final, user_labels_final (최종 스텝 공통 유저와 그 라벨)
# - item_labels_final (최종 스텝 공통 아이템 keep_items에 대한 라벨) 또는 None
# - K_USERS, K_ITEMS

# 0) 보조: 최종 스텝 유저 라벨 딕셔너리 (user_id -> label)
_user_label_map_final = {int(u): int(l) for u, l in zip(ids_final.tolist(), user_labels_final.tolist())}

def _labels_for_user_ids(user_ids: np.ndarray) -> np.ndarray:
    """주어진 user_ids 순서에 맞춰 최종 스텝 라벨을 매핑(없으면 제외하지 않고 -1 할당)."""
    if len(user_ids) == 0:
        return np.empty((0,), dtype=int)
    out = np.full((len(user_ids),), -1, dtype=int)
    for i, u in enumerate(user_ids.tolist()):
        if int(u) in _user_label_map_final:
            out[i] = _user_label_map_final[int(u)]
    return out

def _centers_from_points(points: np.ndarray, labels: np.ndarray, K: int) -> dict:
    """points: (N, d), labels: (N,) -> {k: center or None}"""
    centers = {}
    for k in range(K):
        m = (labels == k)
        centers[k] = points[m].mean(axis=0) if np.any(m) else None
    return centers

def _dist(a, b):
    if a is None or b is None:
        return np.nan
    return float(np.linalg.norm(a - b))

def _pairwise_center_stats(centers: dict):
    """{k: center} -> (mean, min, max) of pairwise distances over existing centers"""
    vecs = [v for v in centers.values() if v is not None]
    if len(vecs) < 2:
        return (np.nan, np.nan, np.nan)
    dists = [np.linalg.norm(a - b) for a, b in combinations(vecs, 2)]
    return (float(np.mean(dists)), float(np.min(dists)), float(np.max(dists)))

rows = []
for si, s in enumerate(STEPS):
    # ---- t-SNE space: user centers ----
    u_st, u_en = slices[s]["user"]
    coords_u = X_tsne[u_st:u_en]                   # (Nu_s, 2)
    ids_u_tsne = ids_for_tsne[s]                   # (Nu_s,)
    labels_u_tsne = _labels_for_user_ids(ids_u_tsne)
    centers_u_tsne = _centers_from_points(coords_u, labels_u_tsne, K_USERS)

    # ---- t-SNE space: item centers ----
    if slices[s]["item"] is not None and item_labels_final is not None:
        i_st, i_en = slices[s]["item"]
        coords_i = X_tsne[i_st:i_en]               # (Ni_common, 2)
        labels_i_tsne = item_labels_final          # keep_items에 대한 라벨(스텝 공통)
        centers_i_tsne = _centers_from_points(coords_i, labels_i_tsne, K_ITEMS)
    else:
        centers_i_tsne = {k: None for k in range(K_ITEMS)}

    # ---- raw space: user centers (원 임베딩) ----
    ids_u_raw = aligned_users_by_step[s]           # (Nr_s,)
    emb_u_raw = user_by_step[s][ids_u_raw, :] if len(ids_u_raw) > 0 else np.empty((0, user_by_step[s].shape[1]))
    labels_u_raw = _labels_for_user_ids(ids_u_raw)
    centers_u_raw = _centers_from_points(emb_u_raw, labels_u_raw, K_USERS)

    # ---- raw space: item centers (원 임베딩; keep_items 부분만 사용) ----
    if item_for_tsne[s] is not None and item_labels_final is not None:
        emb_i_raw = item_for_tsne[s]              # (Ni_common, d)
        labels_i_raw = item_labels_final          # 동일 라벨 사용
        centers_i_raw = _centers_from_points(emb_i_raw, labels_i_raw, K_ITEMS)
    else:
        centers_i_raw = {k: None for k in range(K_ITEMS)}

    # 요약 통계 (K가 2 이상 일반화)
    uu_tsne_mean, uu_tsne_min, uu_tsne_max = _pairwise_center_stats(centers_u_tsne)
    ii_tsne_mean, ii_tsne_min, ii_tsne_max = _pairwise_center_stats(centers_i_tsne)
    uu_raw_mean,  uu_raw_min,  uu_raw_max  = _pairwise_center_stats(centers_u_raw)
    ii_raw_mean,  ii_raw_min,  ii_raw_max  = _pairwise_center_stats(centers_i_raw)

    row = {
        "step": s,
        # user-user / item-item (t-SNE & raw) — 평균/최솟값/최댓값
        "user_dist_tsne_mean": uu_tsne_mean,
        "user_dist_tsne_min":  uu_tsne_min,
        "user_dist_tsne_max":  uu_tsne_max,
        "item_dist_tsne_mean": ii_tsne_mean,
        "item_dist_tsne_min":  ii_tsne_min,
        "item_dist_tsne_max":  ii_tsne_max,
        "user_dist_raw_mean":  uu_raw_mean,
        "user_dist_raw_min":   uu_raw_min,
        "user_dist_raw_max":   uu_raw_max,
        "item_dist_raw_mean":  ii_raw_mean,
        "item_dist_raw_min":   ii_raw_min,
        "item_dist_raw_max":   ii_raw_max,
    }

    # K==2인 경우, 상세(U0/U1, I0/I1, Ux–Iy)도 같이 기록
    if K_USERS == 2:
        row["user_dist_tsne_U0U1"] = _dist(centers_u_tsne.get(0), centers_u_tsne.get(1))
        row["user_dist_raw_U0U1"]  = _dist(centers_u_raw.get(0),  centers_u_raw.get(1))
    if K_ITEMS == 2:
        row["item_dist_tsne_I0I1"] = _dist(centers_i_tsne.get(0), centers_i_tsne.get(1))
        row["item_dist_raw_I0I1"]  = _dist(centers_i_raw.get(0),  centers_i_raw.get(1))
    if K_USERS == 2 and K_ITEMS == 2:
        row.update({
            "ui_tsne_U0_I0": _dist(centers_u_tsne.get(0), centers_i_tsne.get(0)),
            "ui_tsne_U0_I1": _dist(centers_u_tsne.get(0), centers_i_tsne.get(1)),
            "ui_tsne_U1_I0": _dist(centers_u_tsne.get(1), centers_i_tsne.get(0)),
            "ui_tsne_U1_I1": _dist(centers_u_tsne.get(1), centers_i_tsne.get(1)),
            "ui_raw_U0_I0":  _dist(centers_u_raw.get(0),  centers_i_raw.get(0)),
            "ui_raw_U0_I1":  _dist(centers_u_raw.get(0),  centers_i_raw.get(1)),
            "ui_raw_U1_I0":  _dist(centers_u_raw.get(1),  centers_i_raw.get(0)),
            "ui_raw_U1_I1":  _dist(centers_u_raw.get(1),  centers_i_raw.get(1)),
        })

    rows.append(row)

# ---- CSV 저장 ----
dist_csv = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_center_distances_over_steps.csv")
with open(dist_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f"  - saved: {dist_csv}")

# ---- 라인 플롯 1: user-user / item-item (t-SNE vs raw) ----
steps_ax = [r["step"] for r in rows]
ud_t_mean = [r["user_dist_tsne_mean"] for r in rows]
id_t_mean = [r["item_dist_tsne_mean"] for r in rows]
ud_r_mean = [r["user_dist_raw_mean"]  for r in rows]
id_r_mean = [r["item_dist_raw_mean"]  for r in rows]

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(steps_ax, ud_t_mean, marker="o", label="Users (t-SNE, mean)")
axes[0].plot(steps_ax, id_t_mean, marker="o", label="Items (t-SNE, mean)")
axes[0].set_ylabel("Center distance")
axes[0].set_title("User/User & Item/Item center distance over steps (t-SNE)")
axes[0].legend()

axes[1].plot(steps_ax, ud_r_mean, marker="s", label="Users (raw, mean)")
axes[1].plot(steps_ax, id_r_mean, marker="s", label="Items (raw, mean)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Center distance")
axes[1].set_title("User/User & Item/Item center distance over steps (raw)")
axes[1].legend()

plt.tight_layout()
dist_png = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_center_distances_over_steps.png")
plt.savefig(dist_png, dpi=220)
plt.close()
print(f"  - saved: {dist_png}")

# ---- 라인 플롯 2: user-item (K=2일 때 상세), 그 외에는 요약 통계(최소/평균/최대) ----
if K_USERS == 2 and K_ITEMS == 2:
    ui_tsne_u0i0 = [r.get("ui_tsne_U0_I0", np.nan) for r in rows]
    ui_tsne_u0i1 = [r.get("ui_tsne_U0_I1", np.nan) for r in rows]
    ui_tsne_u1i0 = [r.get("ui_tsne_U1_I0", np.nan) for r in rows]
    ui_tsne_u1i1 = [r.get("ui_tsne_U1_I1", np.nan) for r in rows]

    ui_raw_u0i0  = [r.get("ui_raw_U0_I0",  np.nan) for r in rows]
    ui_raw_u0i1  = [r.get("ui_raw_U0_I1",  np.nan) for r in rows]
    ui_raw_u1i0  = [r.get("ui_raw_U1_I0",  np.nan) for r in rows]
    ui_raw_u1i1  = [r.get("ui_raw_U1_I1",  np.nan) for r in rows]

    fig2, axes2 = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    axes2[0].plot(steps_ax, ui_tsne_u0i0, marker="o", label="U0–I0 (t-SNE)")
    axes2[0].plot(steps_ax, ui_tsne_u0i1, marker="o", label="U0–I1 (t-SNE)")
    axes2[0].plot(steps_ax, ui_tsne_u1i0, marker="o", label="U1–I0 (t-SNE)")
    axes2[0].plot(steps_ax, ui_tsne_u1i1, marker="o", label="U1–I1 (t-SNE)")
    axes2[0].set_ylabel("Center distance")
    axes2[0].set_title("User–Item center distances over steps (t-SNE)")
    axes2[0].legend(ncol=2)

    axes2[1].plot(steps_ax, ui_raw_u0i0, marker="s", label="U0–I0 (raw)")
    axes2[1].plot(steps_ax, ui_raw_u0i1, marker="s", label="U0–I1 (raw)")
    axes2[1].plot(steps_ax, ui_raw_u1i0, marker="s", label="U1–I0 (raw)")
    axes2[1].plot(steps_ax, ui_raw_u1i1, marker="s", label="U1–I1 (raw)")
    axes2[1].set_xlabel("Step")
    axes2[1].set_ylabel("Center distance")
    axes2[1].set_title("User–Item center distances over steps (raw)")
    axes2[1].legend(ncol=2)

    plt.tight_layout()
    dist_ui_png = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_user_item_center_distances_over_steps.png")
    plt.savefig(dist_ui_png, dpi=220)
    plt.close()
    print(f"  - saved: {dist_ui_png}")
else:
    # 요약(모든 U_k, I_m 조합의 평균/최소/최대)
    def _ui_stats(centers_u: dict, centers_i: dict):
        vec_u = [v for v in centers_u.values() if v is not None]
        vec_i = [v for v in centers_i.values() if v is not None]
        if len(vec_u) == 0 or len(vec_i) == 0:
            return (np.nan, np.nan, np.nan)
        d = []
        for u in vec_u:
            for i in vec_i:
                d.append(np.linalg.norm(u - i))
        return (float(np.mean(d)), float(np.min(d)), float(np.max(d)))

    ui_tsne_mean = []; ui_tsne_min = []; ui_tsne_max = []
    ui_raw_mean  = []; ui_raw_min  = []; ui_raw_max  = []
    for si, s in enumerate(STEPS):
        u_st, u_en = slices[s]["user"]
        coords_u = X_tsne[u_st:u_en]
        lbl_u = _labels_for_user_ids(ids_for_tsne[s])
        c_u_t = _centers_from_points(coords_u, lbl_u, K_USERS)

        if slices[s]["item"] is not None and item_labels_final is not None:
            i_st, i_en = slices[s]["item"]
            coords_i = X_tsne[i_st:i_en]
            c_i_t = _centers_from_points(coords_i, item_labels_final, K_ITEMS)
        else:
            c_i_t = {k: None for k in range(K_ITEMS)}

        # raw
        ids_u_raw = aligned_users_by_step[s]
        emb_u_raw = user_by_step[s][ids_u_raw, :] if len(ids_u_raw) > 0 else np.empty((0, user_by_step[s].shape[1]))
        lbl_u_raw = _labels_for_user_ids(ids_u_raw)
        c_u_r = _centers_from_points(emb_u_raw, lbl_u_raw, K_USERS)

        if item_for_tsne[s] is not None and item_labels_final is not None:
            emb_i_raw = item_for_tsne[s]
            c_i_r = _centers_from_points(emb_i_raw, item_labels_final, K_ITEMS)
        else:
            c_i_r = {k: None for k in range(K_ITEMS)}

        m, mn, mx = _ui_stats(c_u_t, c_i_t); ui_tsne_mean.append(m); ui_tsne_min.append(mn); ui_tsne_max.append(mx)
        m, mn, mx = _ui_stats(c_u_r, c_i_r); ui_raw_mean.append(m);  ui_raw_min.append(mn);  ui_raw_max.append(mx)

    fig3, axes3 = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    axes3[0].plot(steps_ax, ui_tsne_mean, marker="o", label="mean (t-SNE)")
    axes3[0].plot(steps_ax, ui_tsne_min,  marker="o", label="min (t-SNE)")
    axes3[0].plot(steps_ax, ui_tsne_max,  marker="o", label="max (t-SNE)")
    axes3[0].set_ylabel("Center distance")
    axes3[0].set_title("User–Item center distances over steps (t-SNE)")
    axes3[0].legend()

    axes3[1].plot(steps_ax, ui_raw_mean, marker="s", label="mean (raw)")
    axes3[1].plot(steps_ax, ui_raw_min,  marker="s", label="min (raw)")
    axes3[1].plot(steps_ax, ui_raw_max,  marker="s", label="max (raw)")
    axes3[1].set_xlabel("Step")
    axes3[1].set_ylabel("Center distance")
    axes3[1].set_title("User–Item center distances over steps (raw)")
    axes3[1].legend()

    plt.tight_layout()
    dist_ui_png = os.path.join(OUT_DIR, f"step{CLUSTER_STEP}_user_item_center_distances_over_steps_SUMMARY.png")
    plt.savefig(dist_ui_png, dpi=220)
    plt.close()
    print(f"  - saved: {dist_ui_png}")


In [ ]:
# ===== Clustering metrics per step + common-user matched metrics =====
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import pandas as pd

def compute_metrics(X, y):
    """군집 지표 3종 계산. 표본/군집 부족 시 NaN."""
    X = np.asarray(X); y = np.asarray(y)
    k = len(np.unique(y[y >= 0]))  # 유효 라벨만 카운트
    n = len(y)
    # 유효 표본: y>=0 (라벨 매핑 안 된 사용자 제외)
    mask = (y >= 0)
    Xv, yv = X[mask], y[mask]
    if len(yv) < 3 or k < 2:
        return {"n": int(len(yv)), "k": int(k),
                "silhouette": np.nan,
                "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan}
    return {
        "n": int(len(yv)),
        "k": int(k),
        "silhouette": float(silhouette_score(Xv, yv)),
        "davies_bouldin": float(davies_bouldin_score(Xv, yv)),
        "calinski_harabasz": float(calinski_harabasz_score(Xv, yv)),
    }

# 1) 스텝별(네이티브 표본) 지표: 각 step의 aligned 유저 전부 사용
per_step_rows = []
for s in STEPS:
    ids_s = aligned_users_by_step[s]                         # (Ns,)
    X_s   = user_by_step[s][ids_s, :] if len(ids_s) > 0 else np.empty((0, user_by_step[s].shape[1]))
    y_s   = _labels_for_user_ids(ids_s)                      # 최종 스텝 라벨을 매핑(-1은 미매핑)
    m     = compute_metrics(X_s, y_s)
    per_step_rows.append({"step": s, **m})

df_per_step = pd.DataFrame(per_step_rows).set_index("step").sort_index()
print("\n=== Per-step metrics (native samples per step) ===")
print(df_per_step.to_string())

# 2) 공통 유저 기준 공정 비교: 기준 스텝(CLUSTER_STEP)과 각 step의 교집합으로 동일 사용자만 평가
def subset_by_ids_for_step(step, keep_ids):
    ids = aligned_users_by_step[step]
    X   = user_by_step[step]
    y   = _labels_for_user_ids(ids)
    # ids -> 로컬 인덱스 매핑
    idx_map = {int(u): i for i, u in enumerate(ids.tolist())}
    sel = [idx_map[int(u)] for u in keep_ids if int(u) in idx_map]
    return X[ids[sel], :], y[sel]

common_rows = []
base_ids = aligned_users_by_step[CLUSTER_STEP]
for s in STEPS:
    ids_s   = aligned_users_by_step[s]
    common  = np.intersect1d(base_ids, ids_s)
    X_s, y_s = subset_by_ids_for_step(s, common)
    m = compute_metrics(X_s, y_s)
    common_rows.append({"step": s, "common_users": int(len(common)), **m})

df_common = pd.DataFrame(common_rows).set_index("step").sort_index()
print(f"\n=== Common-user metrics (matched with step{CLUSTER_STEP}) ===")
print(df_common.to_string())

# (선택) 특정 두 스텝 a,b만 직접 비교하고 싶다면:
# a, b = STEPS[0], CLUSTER_STEP
# common_ab = np.intersect1d(aligned_users_by_step[a], aligned_users_by_step[b])
# Xa, ya = subset_by_ids_for_step(a, common_ab)
# Xb, yb = subset_by_ids_for_step(b, common_ab)
# print("\n-- pairwise (a,b) common users:", len(common_ab))
# print("a:", compute_metrics(Xa, ya))
# print("b:", compute_metrics(Xb, yb))


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from typing import Dict, List, Iterable

# ====== 경로 설정 (요청한 파일만 사용) ======
PRED_PATH   = "./Augmentation/data/feedback_loop/predict_label_for_feedback_loop.json"
U_ITEM_PATH = "./A-LLMRec/data/ml-100k/u.item"  # u.item은 1-based movie_id
OUT_DIR     = "./Augmentation/data/feedback_loop/figs_filterbubble"
os.makedirs(OUT_DIR, exist_ok=True)

# ====== 상위 K ======
TOPK = 10

# ====== ML-100K 장르(19) ======
GENRES = [
    "unknown","Action","Adventure","Animation","Children's","Comedy","Crime","Documentary",
    "Drama","Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"
]

def load_u_item(u_item_path: str) -> pd.DataFrame:
    df = pd.read_csv(
        u_item_path, sep='|', header=None, encoding='latin-1',
        names=['movie_id','title','release_date','video_release_date','imdb_url'] + GENRES
    ).set_index('movie_id', drop=True)

    def extract_year(row):
        title = row['title'] if isinstance(row['title'], str) else ""
        m = re.search(r'\((\d{4})\)', title)
        if m: return int(m.group(1))
        rd = row['release_date']
        if isinstance(rd, str):
            m2 = re.search(r'(19|20)\d{2}', rd)
            if m2: return int(m2.group(0))
        return np.nan

    df['year'] = df.apply(extract_year, axis=1)
    return df

def load_predictions(json_path: str) -> Dict[str, List[int]]:
    with open(json_path, 'r') as f:
        raw = json.load(f)
    out: Dict[str, List[int]] = {}
    for k, v in raw.items():
        uid = str(k)
        if isinstance(v, list):
            items = []
            for x in v:
                try: items.append(int(x))
                except: pass
        else:
            items = []
            try: items.append(int(v))
            except: pass
        out[uid] = items
    return out

def resolve_item_ids(pred_ids: Iterable[int], item_df: pd.DataFrame) -> List[int]:
    valid_ids = []
    movie_ids = set(item_df.index.tolist())  # 1..1682
    missing = 0
    for iid in pred_ids:
        if iid in movie_ids:
            valid_ids.append(int(iid))
        elif (iid + 1) in movie_ids:  # 0-based → 1-based 보정
            valid_ids.append(int(iid + 1))
        else:
            missing += 1
    if missing > 0:
        print(f"[WARN] {missing}개 아이템은 u.item에서 찾지 못해 제외했습니다.")
    return valid_ids

def aggregate_counts_all_users(user2pred: Dict[str, List[int]], item_df: pd.DataFrame, topk: int):
    flat_pred = []
    for items in user2pred.values():
        flat_pred.extend(items[:topk])
    if not flat_pred:
        return Counter(), Counter()

    valid_movie_ids = resolve_item_ids(flat_pred, item_df)
    genre_counter, year_counter = Counter(), Counter()
    for mid in valid_movie_ids:
        row = item_df.loc[mid]
        for g in GENRES:
            try:
                if int(row[g]) == 1:
                    genre_counter[g] += 1
            except Exception:
                pass
        y = row.get('year', np.nan)
        if pd.notna(y):
            year_counter[int(y)] += 1
    return genre_counter, year_counter

def plot_bar_from_counter(counter: Counter, title: str, xlabel: str, ylabel: str, outfile: str, sort='key'):
    if not counter:
        print(f"[WARN] No data for {title}, skip plotting.")
        return
    items = list(counter.items())
    if sort == 'key':
        items.sort(key=lambda x: x[0])
    elif sort == 'value_desc':
        items.sort(key=lambda x: x[1], reverse=True)

    labels, values = zip(*items)
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(values)), values)
    plt.xticks(range(len(labels)), labels, rotation=90 if len(labels) > 15 else 45, ha='right')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.close()
    print(f"[SAVE] {outfile}")

def main():
    item_df   = load_u_item(U_ITEM_PATH)
    user2pred = load_predictions(PRED_PATH)

    genre_counter, year_counter = aggregate_counts_all_users(user2pred, item_df, TOPK)

    plot_bar_from_counter(
        genre_counter,
        title=f"Recommended Items by Genre (All Users, Top-{TOPK}/user)",
        xlabel="Genre", ylabel="Count",
        outfile=os.path.join(OUT_DIR, f"genre_counts_top{TOPK}.png"),
        sort='value_desc'
    )
    plot_bar_from_counter(
        year_counter,
        title=f"Recommended Items by Year (All Users, Top-{TOPK}/user)",
        xlabel="Year", ylabel="Count",
        outfile=os.path.join(OUT_DIR, f"year_counts_top{TOPK}.png"),
        sort='key'
    )

    total_items = sum(genre_counter.values())
    print("\n=== Summary ===")
    print(f"- users: {len(user2pred)}")
    print(f"- total counted items (Top-{TOPK} per user, after mapping): {total_items}")
    if total_items > 0:
        print("Top genres:", sorted(genre_counter.items(), key=lambda x: x[1], reverse=True)[:10])

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from typing import List

# ===================== 설정 =====================
PRED_PATH   = "./Augmentation/data/feedback_loop/predict_label_for_feedback_loop.json"
U_ITEM_PATH = "./A-LLMRec/data/ml-100k/u.item"   # u.item은 1-based movie_id
OUT_DIR     = "./Augmentation/data/feedback_loop/filterbubble_figs"
TOPK        = 10
os.makedirs(OUT_DIR, exist_ok=True)

# =============== ML-100K 장르(19개) ===============
GENRES = [
    "unknown","Action","Adventure","Animation","Children's","Comedy","Crime","Documentary",
    "Drama","Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"
]

# =============== u.item 로드 ===============
def load_u_item(u_item_path: str) -> pd.DataFrame:
    df = pd.read_csv(
        u_item_path,
        sep='|', header=None, encoding='latin-1',
        names=['movie_id','title','release_date','video_release_date','imdb_url'] + GENRES,
        engine='python'
    ).set_index('movie_id', drop=True)
    return df  # index = movie_id(1-based)

# =============== ID 보정: (0-based) → (u.item의 1-based) ===============
def to_movie_ids(pred_ids: List[int], u_item_df: pd.DataFrame) -> List[int]:
    valid = []
    movie_ids = set(u_item_df.index.tolist())  # 1..1682
    for x in pred_ids:
        if x in movie_ids:
            valid.append(int(x))
        elif (x + 1) in movie_ids:  # Aug 0-based → +1
            valid.append(int(x + 1))
        # else: drop
    return valid

# =============== 사용자별 Top-K 동일 장르 최빈수 계산 ===============
def max_same_genre_count_for(user2preds: dict, u_item_df: pd.DataFrame, topk: int) -> List[int]:
    counts = []
    for uid, pred_ids in user2preds.items():
        mids = to_movie_ids(pred_ids[:topk], u_item_df)
        if not mids:
            counts.append(0); continue
        g_cnt = Counter()
        for mid in mids:
            if mid not in u_item_df.index: 
                continue
            row = u_item_df.loc[mid]
            for g in GENRES:
                try:
                    if int(row[g]) == 1:
                        g_cnt[g] += 1
                except:
                    pass
        mx = max(g_cnt.values()) if g_cnt else 0
        if mx == 0 and len(mids) > 0:
            mx = 1
        counts.append(min(mx, topk))
    return counts

# =============== 히스토그램 저장 ===============
def plot_hist(counts: List[int], topk: int, outfile: str, title_suffix: str = ""):
    dist = Counter(counts)
    xs = list(range(1, topk+1))
    ys = [dist.get(x, 0) for x in xs]
    plt.figure(figsize=(10, 5))
    plt.bar(xs, ys)
    plt.xticks(xs)
    plt.xlabel(f"Max same-genre count in Top-{topk}")
    plt.ylabel("#users")
    title = f"Top-{topk} genre concentration per user"
    if title_suffix: title += f" ({title_suffix})"
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.close()
    print(f"[SAVE] {outfile}")

# ===================== 실행 =====================
def run():
    # 예측 파일 하나만 읽기
    with open(PRED_PATH, "r") as f:
        raw = json.load(f)  # { "uid": [item_id, ...], ... }
    # 키/값 정규화
    user2preds = {}
    for k, v in raw.items():
        uid = str(k)
        if isinstance(v, list):
            ids = []
            for x in v:
                try: ids.append(int(x))
                except: pass
        else:
            ids = []
            try: ids.append(int(v))
            except: pass
        user2preds[uid] = ids

    # 메타 로드 → 장르 매핑
    u_item_df = load_u_item(U_ITEM_PATH)

    # 동일 장르 최빈수 분포 계산 & 저장
    counts = max_same_genre_count_for(user2preds, u_item_df, TOPK)
    out_png = os.path.join(OUT_DIR, f"predict_label_for_feedback_loop__max_same_genre_top{TOPK}.png")
    plot_hist(counts, TOPK, out_png, title_suffix="predict_label_for_feedback_loop.json")

    # (선택) 요약 통계 출력
    arr = np.array(counts, dtype=int)
    if arr.size:
        print("\nSummary:")
        print(f"- users: {arr.size}")
        print(f"- mean max-same-genre: {arr.mean():.3f}")
        print(f"- median: {np.median(arr):.3f}")
        print(f"- min/max: {arr.min()} / {arr.max()}")
    else:
        print("[WARN] No users or empty predictions.")

if __name__ == "__main__":
    run()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from typing import Dict, List
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE  # (t-SNE는 클러스터에는 직접 필요 없지만 의존성 유지)

# =========================
# 경로/설정
# =========================
AUG_BASE = "./Augmentation/data/feedback_loop"
PARTS    = 5                                   # step 수 (0..PARTS-1)
STEPS    = list(range(PARTS))
CLUSTER_STEP = STEPS[-1]                        # 최종 step에서 KMeans 수행
OUT_DIR  = os.path.join(AUG_BASE, "filterbubble_figs")
os.makedirs(OUT_DIR, exist_ok=True)

# 추천 결과(유저별 candidate item list)
PRED_PATH    = os.path.join(AUG_BASE, "predict_label_for_feedback_loop.json")
# u.item은 1-based movie_id (장르/연도 매핑용)
U_ITEM_PATH  = "./A-LLMRec/data/ml-100k/u.item"

# 분석 옵션
K_USERS = 2
TOPK    = 10

# =========================
# 유틸
# =========================
def load_users_items_for_step(base_dir: str, parts: int, step: int):
    upath = os.path.join(base_dir, f"part5/try1/embeddings_users_part{parts}_step{step}.npy")
    ipath = os.path.join(base_dir, f"part5/try1/embeddings_items_part{parts}_step{step}.npy")
    if not os.path.exists(upath):
        raise FileNotFoundError(upath)
    U = np.load(upath)  # (num_users, d)
    I = np.load(ipath) if os.path.exists(ipath) else None
    if U.ndim != 2:
        raise ValueError(f"user_emb shape invalid at step{step}: {U.shape}")
    if I is not None and I.ndim != 2:
        raise ValueError(f"item_emb shape invalid at step{step}: {I.shape}")
    return U, I

def load_common_users_from_json(base_dir: str):
    # 사용자가 준 코드와 동일: train_raw.json ∩ label.json
    train_path = os.path.join(base_dir, "train_raw.json")
    label_path = os.path.join(base_dir, "label.json")
    with open(train_path, "r") as f:
        train = json.load(f)
    with open(label_path, "r") as f:
        label = json.load(f)
    train_users = set(map(int, train.keys()))
    label_users = set(map(int, label.keys()))
    return np.array(sorted(train_users & label_users), dtype=np.int64)

def align_users_for_step(common_users: np.ndarray, num_users_step: int):
    if len(common_users) == 0:
        return np.array([], dtype=np.int64)
    mask = (common_users >= 0) & (common_users < num_users_step)
    return common_users[mask]

# ====== ML-100K 장르(19개) ======
GENRES = [
    "unknown","Action","Adventure","Animation","Children's","Comedy","Crime","Documentary",
    "Drama","Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"
]

def load_u_item(u_item_path: str) -> pd.DataFrame:
    df = pd.read_csv(
        u_item_path,
        sep='|', header=None, encoding='latin-1',
        names=['movie_id','title','release_date','video_release_date','imdb_url'] + GENRES,
        engine='python'
    ).set_index('movie_id', drop=True)

    def extract_year(row):
        title = row['title'] if isinstance(row['title'], str) else ""
        m = re.search(r'\((\d{4})\)', title)
        if m: return int(m.group(1))
        rd = row['release_date']
        if isinstance(rd, str):
            m2 = re.search(r'(19|20)\d{2}', rd)
            if m2: return int(m2.group(0))
        return np.nan

    df['year'] = df.apply(extract_year, axis=1)
    return df

def load_predictions(json_path: str) -> Dict[str, List[int]]:
    # { "uid": [item_id, ...], ... }
    with open(json_path, "r") as f:
        raw = json.load(f)
    user2preds = {}
    for k, v in raw.items():
        uid = str(k)
        if isinstance(v, list):
            ids = []
            for x in v:
                try: ids.append(int(x))
                except: pass
        else:
            ids = []
            try: ids.append(int(v))
            except: pass
        user2preds[uid] = ids
    return user2preds

# =============== ID 보정: (Aug 0-based) → (u.item 1-based) ===============
def to_movie_ids(pred_ids: List[int], u_item_df: pd.DataFrame) -> List[int]:
    valid = []
    movie_ids = set(u_item_df.index.tolist())  # 1..1682
    for x in pred_ids:
        if x in movie_ids:
            valid.append(int(x))
        elif (x + 1) in movie_ids:  # Aug 0-based → +1
            valid.append(int(x + 1))
        # else drop
    return valid

# =============== 사용자별 Top-K 동일 장르 최빈수 계산 ===============
def max_same_genre_count_for(user2preds: Dict[str, List[int]],
                             u_item_df: pd.DataFrame,
                             topk: int) -> List[int]:
    counts = []
    for uid, pred_ids in user2preds.items():
        mids = to_movie_ids(pred_ids[:topk], u_item_df)
        if not mids:
            counts.append(0); continue
        g_cnt = Counter()
        for mid in mids:
            if mid not in u_item_df.index:
                continue
            row = u_item_df.loc[mid]
            for g in GENRES:
                try:
                    if int(row[g]) == 1:
                        g_cnt[g] += 1
                except:
                    pass
        mx = max(g_cnt.values()) if g_cnt else 0
        if mx == 0 and len(mids) > 0:
            mx = 1
        counts.append(min(mx, topk))
    return counts

def plot_hist(counts: List[int], topk: int, outfile: str, title: str):
    dist = Counter(counts)
    xs = list(range(1, topk+1))
    ys = [dist.get(x, 0) for x in xs]
    plt.figure(figsize=(8, 4.5))
    plt.bar(xs, ys)
    plt.xticks(xs)
    plt.xlabel(f"Max same-genre count in Top-{topk}")
    plt.ylabel("#users")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.close()
    print(f"[SAVE] {outfile}")

# =========================
# 파이프라인
# =========================
def main():
    # 1) 임베딩 로드 및 차원 체크
    print("Loading augmentation embeddings per step...")
    user_by_step, item_by_step = {}, {}
    dims = set()
    for s in STEPS:
        U, I = load_users_items_for_step(AUG_BASE, PARTS, s)
        user_by_step[s], item_by_step[s] = U, I
        dims.add(U.shape[1])
        if I is not None: dims.add(I.shape[1])
        print(f"  - step{s}: users={U.shape}, items={None if I is None else I.shape}")
    if len(dims) != 1:
        raise RuntimeError(f"임베딩 차원이 step별로 다릅니다: {dims}")
    D = dims.pop()

    # 2) 공통 유저 정렬
    print("Compute base common users from train ∩ label")
    common_users_base = load_common_users_from_json(AUG_BASE)
    if len(common_users_base) == 0:
        raise RuntimeError("train ∩ label 공통 유저가 없습니다.")
    aligned_users_by_step = {s: align_users_for_step(common_users_base, user_by_step[s].shape[0])
                             for s in STEPS}

    # 3) 최종 step에서 KMeans → 사용자 레이블 생성
    print(f"KMeans on final step: step{CLUSTER_STEP}")
    U_final = user_by_step[CLUSTER_STEP]
    ids_final = aligned_users_by_step[CLUSTER_STEP]
    if len(ids_final) < K_USERS:
        raise RuntimeError(f"최종 step에서 클러스터링 가능한 공통 유저가 부족합니다: {len(ids_final)}")
    emb_final_users = U_final[ids_final, :]  # (|ids_final|, D)
    kmeans = KMeans(n_clusters=K_USERS, random_state=42, n_init=10)
    user_labels_final = kmeans.fit_predict(emb_final_users)  # (|ids_final|,)

    # 4) user_id → cluster 레이블 맵
    user_cluster = {str(int(u)): int(c) for u, c in zip(ids_final.tolist(), user_labels_final.tolist())}

    # 5) 추천 로드
    user2preds = load_predictions(PRED_PATH)

    # 6) 클러스터별 사용자 분할(예측이 있는 유저만)
    users_c = {k: [] for k in range(K_USERS)}
    for uid_str, cl in user_cluster.items():
        if uid_str in user2preds:
            users_c[cl].append(uid_str)
    # 레이블 없는/예측 미보유 유저는 제외
    unlabeled = [uid for uid in user2preds.keys() if uid not in user_cluster]
    if unlabeled:
        print(f"[INFO] users missing cluster label or not in common set: {len(unlabeled)} (ignored)")

    # 7) 장르/연도 매핑 데이터 로드
    u_item_df = load_u_item(U_ITEM_PATH)

    # 8) 클러스터별 동장르 최빈수 계산 & 저장
    for c in range(K_USERS):
        sub = {u: user2preds[u] for u in users_c[c]}
        counts = max_same_genre_count_for(sub, u_item_df, TOPK) if sub else []
        if counts:
            plot_hist(
                counts, TOPK,
                os.path.join(OUT_DIR, f"cluster{c}_max_same_genre_top{TOPK}.png"),
                title=f"Cluster {c} — Top-{TOPK} genre concentration"
            )
        print(f"Cluster {c}: users={len(users_c[c])}, "
              f"mean={np.mean(counts) if counts else np.nan:.3f}, "
              f"median={np.median(counts) if counts else np.nan:.3f}")

    # 9) 비교 그림(0/1 나란히)
    if K_USERS == 2:
        xs = list(range(1, TOPK+1))
        c0 = max_same_genre_count_for({u: user2preds[u] for u in users_c[0]}, u_item_df, TOPK) if users_c[0] else []
        c1 = max_same_genre_count_for({u: user2preds[u] for u in users_c[1]}, u_item_df, TOPK) if users_c[1] else []
        dist0, dist1 = Counter(c0), Counter(c1)
        ys0 = [dist0.get(x, 0) for x in xs]
        ys1 = [dist1.get(x, 0) for x in xs]

        plt.figure(figsize=(9, 4.8))
        width = 0.4
        x0 = np.array(xs) - width/2
        x1 = np.array(xs) + width/2
        plt.bar(x0, ys0, width=width, label=f"Cluster 0 (n={len(c0)})")
        plt.bar(x1, ys1, width=width, label=f"Cluster 1 (n={len(c1)})")
        plt.xticks(xs)
        plt.xlabel(f"Max same-genre count in Top-{TOPK}")
        plt.ylabel("#users")
        plt.title(f"Top-{TOPK} genre concentration — by cluster (step{CLUSTER_STEP})")
        plt.legend()
        plt.tight_layout()
        out_cmp = os.path.join(OUT_DIR, f"cluster01_compare_max_same_genre_top{TOPK}.png")
        plt.savefig(out_cmp, dpi=300)
        plt.close()
        print(f"[SAVE] {out_cmp}")

if __name__ == "__main__":
    main()
